# 000 - Preparar dados brutos

Este notebook prepara a pasta de entrada para o `010_extracao_transformacao.ipynb`.

Ele le pastas no padrao `dados_gaia_ref_YYYYMM`, seleciona a planilha mais recente por politica publica, copia os arquivos selecionados para `dados_brutos/dado_atual/dados_atuais` e gera um inventario rastreavel da preparacao.

O notebook nao transforma indicadores, nao altera dados internos das planilhas e nao substitui o ETL do `010`.

In [10]:
from datetime import datetime
from pathlib import Path
import hashlib
import os
import re
import shutil
import unicodedata

import pandas as pd

## Parametros

- `MODO_ESTRITO = True`: exige que todas as politicas obrigatorias venham da competencia mais recente.
- `MODO_ESTRITO = False`: permite completar lacunas com pastas anteriores, registrando a mistura no inventario.
- `DADOS_BRUTOS_ORIGEM_ROOT`: variavel de ambiente opcional para apontar outra raiz de leitura.

In [11]:
MODO_ESTRITO = False
PARAR_SE_FALTAR_OBRIGATORIA = True
LIMPAR_PLANILHAS_DESTINO = True
COPIAR_POLITICAS_OPCIONAIS = False

ORIGEM_ROOT_OVERRIDE = os.environ.get("DADOS_BRUTOS_ORIGEM_ROOT") or None
PASTA_DESTINO_NOME = "dados_atuais"
PADRAO_PASTA_REFERENCIA = r"^dados_gaia_ref_(\d{6})$"

CSV_SEP = ";"
CSV_ENCODING = "utf-8-sig"
RUN_TIMESTAMP = datetime.now()
RUN_DATETIME = RUN_TIMESTAMP.strftime("%Y%m%d%H%M%S")

print(f"Data/hora da preparacao: {RUN_TIMESTAMP:%d/%m/%Y %H:%M:%S}")
print(f"Modo estrito: {MODO_ESTRITO}")
print(f"Copiar opcionais: {COPIAR_POLITICAS_OPCIONAIS}")

Data/hora da preparacao: 14/05/2026 16:35:44
Modo estrito: False
Copiar opcionais: False


In [12]:
inicio = Path.cwd().resolve()
PROJECT_ROOT = inicio

for candidato in [inicio, *inicio.parents]:
    tem_notebooks = (candidato / "notebooks").exists()
    tem_requirements = (candidato / "requirements.txt").exists()
    tem_config = (candidato / "config" / "ufs.csv").exists()
    if tem_notebooks and (tem_requirements or tem_config):
        PROJECT_ROOT = candidato
        break

DADO_ATUAL_ROOT = PROJECT_ROOT / "dados_brutos" / "dado_atual"
ORIGEM_ROOT = Path(ORIGEM_ROOT_OVERRIDE).expanduser().resolve() if ORIGEM_ROOT_OVERRIDE else DADO_ATUAL_ROOT
DESTINO_DIR = DADO_ATUAL_ROOT / PASTA_DESTINO_NOME
DESTINO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Raiz de leitura: {ORIGEM_ROOT}")
print(f"Pasta preparada: {DESTINO_DIR}")

if not ORIGEM_ROOT.exists():
    raise FileNotFoundError(f"Raiz de leitura nao encontrada: {ORIGEM_ROOT}")

Raiz do projeto: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei
Raiz de leitura: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual
Pasta preparada: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual\dados_atuais


In [13]:
POLITICAS = [
    {"politica": "caf_dap", "rotulo": "CAF/DAP", "obrigatoria": True, "regex": r"\b(caf|dap)\b|caf_dap"},
    {"politica": "ater", "rotulo": "ATER", "obrigatoria": True, "regex": r"\bater\b|ater_"},
    {"politica": "pronaf", "rotulo": "PRONAF", "obrigatoria": True, "regex": r"\bpronaf\b|pronaf_"},
    {"politica": "mais_alimentos", "rotulo": "Mais Alimentos", "obrigatoria": True, "regex": r"mais[_ -]?alimentos"},
    {"politica": "garantia_safra", "rotulo": "Garantia-Safra", "obrigatoria": True, "regex": r"garantia[_ -]?safra"},
    {"politica": "pncf", "rotulo": "PNCF", "obrigatoria": True, "regex": r"\bpncf\b|pncf_"},
    {"politica": "pnra", "rotulo": "PNRA", "obrigatoria": True, "regex": r"\bpnra\b|pnra_"},
    {"politica": "selo_bio", "rotulo": "Selo Bio", "obrigatoria": False, "regex": r"selo.*bio|bio.*selo"},
]

df_politicas = pd.DataFrame(POLITICAS)
display(df_politicas[["politica", "rotulo", "obrigatoria", "regex"]])

,politica,rotulo,obrigatoria,regex
0,caf_dap,CAF/DAP,True,\b(caf|dap)\b|caf_dap
1,ater,ATER,True,\bater\b|ater_
2,pronaf,PRONAF,True,\bpronaf\b|pronaf_
3,mais_alimentos,Mais Alimentos,True,mais[_ -]?alimentos
4,garantia_safra,Garantia-Safra,True,garantia[_ -]?safra
5,pncf,PNCF,True,\bpncf\b|pncf_
6,pnra,PNRA,True,\bpnra\b|pnra_
7,selo_bio,Selo Bio,False,selo.*bio|bio.*selo


In [14]:
def normalizar_texto(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(ch for ch in texto if not unicodedata.combining(ch))
    return texto.lower().strip()


def competencia_from_nome_pasta(nome):
    match = re.match(PADRAO_PASTA_REFERENCIA, str(nome))
    return match.group(1) if match else None


def competencia_from_texto(valor):
    if valor is None or pd.isna(valor):
        return None
    texto = str(valor)
    match = re.search(r"(20\d{2})\D?(0[1-9]|1[0-2])", texto)
    if match:
        return f"{match.group(1)}{match.group(2)}"
    match = re.search(r"(20\d{2})(0[1-9]|1[0-2])", texto)
    if match:
        return f"{match.group(1)}{match.group(2)}"
    return None


def classificar_politica(arquivo):
    nome = normalizar_texto(arquivo.name)
    for item in POLITICAS:
        if re.search(item["regex"], nome):
            return item
    return None


def primeiro_valor_nao_nulo(df, aliases):
    colunas_normalizadas = {normalizar_texto(col): col for col in df.columns}
    for alias in aliases:
        coluna = colunas_normalizadas.get(alias)
        if coluna is None:
            continue
        serie = df[coluna].dropna()
        if not serie.empty:
            return str(serie.iloc[0])
    return None


def extrair_metadados_excel(caminho):
    info = {
        "dt_referencia": None,
        "dt_geracao": None,
        "competencia_dado": None,
        "aba_metadados": None,
        "erro_metadados": None,
    }
    try:
        xls = pd.ExcelFile(caminho, engine="openpyxl")
        prioridade = ["DADOS", "Dados", "TOTALIZADORES", "Totalizadores"]
        abas = []
        for aba in prioridade + list(xls.sheet_names):
            if aba in xls.sheet_names and aba not in abas:
                abas.append(aba)

        for aba in abas:
            try:
                amostra = pd.read_excel(xls, sheet_name=aba, nrows=10)
            except Exception:
                continue
            if info["dt_referencia"] is None:
                info["dt_referencia"] = primeiro_valor_nao_nulo(amostra, ["dt_referencia", "data_referencia", "referencia"])
            if info["dt_geracao"] is None:
                info["dt_geracao"] = primeiro_valor_nao_nulo(amostra, ["dt_geracao", "data_geracao", "geracao"])
            if info["dt_referencia"] or info["dt_geracao"]:
                info["aba_metadados"] = aba
            if info["dt_referencia"] and info["dt_geracao"]:
                break
    except Exception as exc:
        info["erro_metadados"] = str(exc)

    info["competencia_dado"] = competencia_from_texto(info["dt_referencia"]) or competencia_from_texto(caminho.name)
    return info


def hash_sha256(caminho):
    h = hashlib.sha256()
    with open(caminho, "rb") as fh:
        for bloco in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

In [15]:
if competencia_from_nome_pasta(ORIGEM_ROOT.name):
    pastas_referencia = [{"competencia_pasta": competencia_from_nome_pasta(ORIGEM_ROOT.name), "pasta": ORIGEM_ROOT}]
else:
    pastas_referencia = []
    for pasta in ORIGEM_ROOT.iterdir():
        if not pasta.is_dir():
            continue
        competencia = competencia_from_nome_pasta(pasta.name)
        if competencia:
            pastas_referencia.append({"competencia_pasta": competencia, "pasta": pasta})

pastas_referencia = sorted(pastas_referencia, key=lambda row: row["competencia_pasta"], reverse=True)
if not pastas_referencia:
    raise FileNotFoundError(f"Nenhuma pasta no padrao dados_gaia_ref_YYYYMM encontrada em {ORIGEM_ROOT}")

competencia_alvo = pastas_referencia[0]["competencia_pasta"]
print(f"Competencia mais recente encontrada: {competencia_alvo}")
display(pd.DataFrame([{**row, "pasta": str(row["pasta"])} for row in pastas_referencia]))

Competencia mais recente encontrada: 202604


,competencia_pasta,pasta
0,202604,C:\Users\marce\OneDrive - Ministério da Agricu...
1,202603,C:\Users\marce\OneDrive - Ministério da Agricu...


In [16]:
candidatos = []
nao_classificados = []

for row_pasta in pastas_referencia:
    pasta = row_pasta["pasta"]
    competencia_pasta = row_pasta["competencia_pasta"]
    for arquivo in sorted(pasta.glob("*.xlsx")):
        politica = classificar_politica(arquivo)
        if politica is None:
            nao_classificados.append({
                "arquivo": arquivo.name,
                "pasta_origem": str(pasta),
                "competencia_pasta": competencia_pasta,
            })
            continue
        metadados = extrair_metadados_excel(arquivo)
        stat = arquivo.stat()
        competencia_nome_arquivo = competencia_from_texto(arquivo.name)
        competencia_ordenacao = metadados["competencia_dado"] or competencia_pasta or competencia_nome_arquivo or "000000"
        candidatos.append({
            "politica": politica["politica"],
            "rotulo": politica["rotulo"],
            "obrigatoria": politica["obrigatoria"],
            "arquivo_origem": arquivo.name,
            "caminho_origem": str(arquivo),
            "pasta_origem": str(pasta),
            "competencia_pasta": competencia_pasta,
            "competencia_nome_arquivo": competencia_nome_arquivo,
            "competencia_dado": metadados["competencia_dado"],
            "competencia_ordenacao": competencia_ordenacao,
            "dt_referencia": metadados["dt_referencia"],
            "dt_geracao": metadados["dt_geracao"],
            "aba_metadados": metadados["aba_metadados"],
            "erro_metadados": metadados["erro_metadados"],
            "mtime_origem": datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
            "tamanho_bytes": stat.st_size,
        })

df_candidatos = pd.DataFrame(candidatos)
df_nao_classificados = pd.DataFrame(nao_classificados)

print(f"Arquivos candidatos: {df_candidatos.shape[0]}")
print(f"Arquivos nao classificados: {df_nao_classificados.shape[0]}")
if not df_candidatos.empty:
    display(df_candidatos.sort_values(["politica", "competencia_ordenacao", "competencia_pasta"], ascending=[True, False, False]))
if not df_nao_classificados.empty:
    display(df_nao_classificados)

Arquivos candidatos: 13
Arquivos nao classificados: 0


,politica,rotulo,obrigatoria,arquivo_origem,caminho_origem,pasta_origem,competencia_pasta,competencia_nome_arquivo,competencia_dado,competencia_ordenacao,dt_referencia,dt_geracao,aba_metadados,erro_metadados,mtime_origem,tamanho_bytes
0,ater,ATER,True,ater_ate_2026_04_gerado_em_20260512153352.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,20260512,DADOS,None,2026-05-12 15:37:01,29152
6,ater,ATER,True,ater_ate_2026_03_gerado_em_20260410151127.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202603,202603,202603,2026_03,20260410,DADOS,None,2026-05-05 12:48:06,32701
1,caf_dap,CAF/DAP,True,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,2026_05_12,DADOS,None,2026-05-12 15:55:14,278250
7,caf_dap,CAF/DAP,True,caf_dap_ativos_ate_2026_03_gerado_em_202604101...,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202603,202603,202603,2026_03,2026_04_10,DADOS,None,2026-05-05 12:48:06,322233
8,garantia_safra,Garantia-Safra,True,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,2023-2024,2026_04_13,DADOS,None,2026-05-05 12:48:06,96842
2,mais_alimentos,Mais Alimentos,True,mais_alimentos_gaia_202605131504.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202605,202604,202604,2026_04,2026_05_13,Dados,None,2026-05-13 15:04:44,203644
9,mais_alimentos,Mais Alimentos,True,mais_alimentos_gaia_202604151554.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202603,202603,2026_03,2026_04_14,Dados,None,2026-05-05 12:48:06,232701
3,pncf,PNCF,True,pncf_2026_04_gerado_em_20260514095754.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,2026_05_14,DADOS,None,2026-05-14 09:57:54,13922
10,pncf,PNCF,True,pncf_2026_03_gerado_em_20260410170006.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202603,202603,202603,2026_03,2026_04_10,DADOS,None,2026-05-05 12:48:06,21330
11,pnra,PNRA,True,PNRA_2026_2026_04_15.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,2026 até mar,2026_04_15,DADOS,None,2026-05-05 12:48:06,306820


In [17]:
selecionados = []

for politica in POLITICAS:
    if not politica["obrigatoria"] and not COPIAR_POLITICAS_OPCIONAIS:
        continue

    if df_candidatos.empty:
        subset = df_candidatos
    else:
        subset = df_candidatos[df_candidatos["politica"] == politica["politica"]].copy()
        if MODO_ESTRITO:
            subset = subset[subset["competencia_pasta"] == competencia_alvo].copy()

    if subset.empty:
        selecionados.append({
            "politica": politica["politica"],
            "rotulo": politica["rotulo"],
            "obrigatoria": politica["obrigatoria"],
            "status": "faltante",
            "observacao": "Arquivo obrigatorio nao encontrado" if politica["obrigatoria"] else "Arquivo opcional nao encontrado",
        })
        continue

    subset = subset.sort_values(
        ["competencia_ordenacao", "competencia_pasta", "mtime_origem", "arquivo_origem"],
        ascending=[False, False, False, True],
    )
    escolhido = subset.iloc[0].to_dict()
    escolhido["status"] = "selecionado"
    escolhido["observacao"] = "Selecionado por competencia do dado, pasta, nome do arquivo e data de modificacao"
    selecionados.append(escolhido)

df_selecionados = pd.DataFrame(selecionados)
display(df_selecionados)

,politica,rotulo,obrigatoria,arquivo_origem,caminho_origem,pasta_origem,competencia_pasta,competencia_nome_arquivo,competencia_dado,competencia_ordenacao,dt_referencia,dt_geracao,aba_metadados,erro_metadados,mtime_origem,tamanho_bytes,status,observacao
0,caf_dap,CAF/DAP,True,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,2026_05_12,DADOS,None,2026-05-12 15:55:14,278250,selecionado,"Selecionado por competencia do dado, pasta, no..."
1,ater,ATER,True,ater_ate_2026_04_gerado_em_20260512153352.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,20260512,DADOS,None,2026-05-12 15:37:01,29152,selecionado,"Selecionado por competencia do dado, pasta, no..."
2,pronaf,PRONAF,True,pronaf_gaia_202605131604.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202605,202604,202604,2026_04,2026_05_13,Dados,None,2026-05-13 16:10:20,680350,selecionado,"Selecionado por competencia do dado, pasta, no..."
3,mais_alimentos,Mais Alimentos,True,mais_alimentos_gaia_202605131504.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202605,202604,202604,2026_04,2026_05_13,Dados,None,2026-05-13 15:04:44,203644,selecionado,"Selecionado por competencia do dado, pasta, no..."
4,garantia_safra,Garantia-Safra,True,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,2023-2024,2026_04_13,DADOS,None,2026-05-05 12:48:06,96842,selecionado,"Selecionado por competencia do dado, pasta, no..."
5,pncf,PNCF,True,pncf_2026_04_gerado_em_20260514095754.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,2026_04,2026_05_14,DADOS,None,2026-05-14 09:57:54,13922,selecionado,"Selecionado por competencia do dado, pasta, no..."
6,pnra,PNRA,True,PNRA_2026_2026_04_15.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,2026 até mar,2026_04_15,DADOS,None,2026-05-05 12:48:06,306820,selecionado,"Selecionado por competencia do dado, pasta, no..."


In [18]:
linhas_inventario = []
faltantes_obrigatorios = []

for _, row in df_selecionados.iterrows():
    item = row.to_dict()
    item["arquivo_destino"] = ""
    item["caminho_destino"] = ""
    item["hash_sha256"] = ""
    item["preparado_em"] = RUN_TIMESTAMP.strftime("%Y-%m-%d %H:%M:%S")
    if item.get("status") == "faltante" and item.get("obrigatoria"):
        faltantes_obrigatorios.append(item.get("rotulo") or item.get("politica"))
    linhas_inventario.append(item)

df_inventario = pd.DataFrame(linhas_inventario)
inventario_timestamp = DESTINO_DIR / f"inventario_preparacao_{RUN_DATETIME}.csv"
inventario_atual = DESTINO_DIR / "inventario_preparacao_atual.csv"

if LIMPAR_PLANILHAS_DESTINO:
    for padrao in ["*.xlsx", "inventario_preparacao_*.csv", "inventario_preparacao_atual.csv"]:
        for arquivo in DESTINO_DIR.glob(padrao):
            arquivo.unlink()

if faltantes_obrigatorios and PARAR_SE_FALTAR_OBRIGATORIA:
    df_inventario.to_csv(inventario_timestamp, sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
    df_inventario.to_csv(inventario_atual, sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
    display(df_inventario)
    raise FileNotFoundError(f"Politicas obrigatorias sem arquivo selecionado: {faltantes_obrigatorios}")

for idx, row in df_inventario[df_inventario["status"] == "selecionado"].iterrows():
    origem = Path(row["caminho_origem"])
    destino = DESTINO_DIR / origem.name
    shutil.copy2(origem, destino)
    df_inventario.loc[idx, "arquivo_destino"] = destino.name
    df_inventario.loc[idx, "caminho_destino"] = str(destino)
    df_inventario.loc[idx, "hash_sha256"] = hash_sha256(destino)

df_inventario.to_csv(inventario_timestamp, sep=CSV_SEP, index=False, encoding=CSV_ENCODING)
df_inventario.to_csv(inventario_atual, sep=CSV_SEP, index=False, encoding=CSV_ENCODING)

competencias_obrigatorias = sorted(
    set(
        df_inventario.loc[
            (df_inventario["status"] == "selecionado") & (df_inventario["obrigatoria"] == True),
            "competencia_pasta",
        ].dropna().astype(str)
    )
)

print(f"Inventario salvo em: {inventario_timestamp}")
print(f"Inventario atual salvo em: {inventario_atual}")
print(f"Planilhas preparadas em: {DESTINO_DIR}")
print(f"Competencias usadas nas politicas obrigatorias: {competencias_obrigatorias}")

if len(competencias_obrigatorias) > 1:
    print("Atencao: a pasta preparada mistura competencias de origem. Verifique o inventario antes de rodar o 010.")

display(df_inventario)

Inventario salvo em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual\dados_atuais\inventario_preparacao_20260514163544.csv
Inventario atual salvo em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual\dados_atuais\inventario_preparacao_atual.csv
Planilhas preparadas em: C:\Users\marce\OneDrive - Ministério da Agricultura e Pecuária\LAB_DATA_MDA\20260424_relatorio_dados\relatorio_dados_damei\dados_brutos\dado_atual\dados_atuais
Competencias usadas nas politicas obrigatorias: ['202603', '202604']
Atencao: a pasta preparada mistura competencias de origem. Verifique o inventario antes de rodar o 010.


,politica,rotulo,obrigatoria,arquivo_origem,caminho_origem,pasta_origem,competencia_pasta,competencia_nome_arquivo,competencia_dado,competencia_ordenacao,...,aba_metadados,erro_metadados,mtime_origem,tamanho_bytes,status,observacao,arquivo_destino,caminho_destino,hash_sha256,preparado_em
0,caf_dap,CAF/DAP,True,caf_dap_ativos_ate_2026_04_gerado_em_202605121...,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,...,DADOS,None,2026-05-12 15:55:14,278250,selecionado,"Selecionado por competencia do dado, pasta, no...",caf_dap_ativos_ate_2026_04_gerado_em_202605121...,C:\Users\marce\OneDrive - Ministério da Agricu...,273dd302d2121f9449a8f0efc29101188949fd7f647641...,2026-05-14 16:35:44
1,ater,ATER,True,ater_ate_2026_04_gerado_em_20260512153352.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,...,DADOS,None,2026-05-12 15:37:01,29152,selecionado,"Selecionado por competencia do dado, pasta, no...",ater_ate_2026_04_gerado_em_20260512153352.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,47b437b36329471b7b5668d0fcb6d5c122b52705d961cf...,2026-05-14 16:35:44
2,pronaf,PRONAF,True,pronaf_gaia_202605131604.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202605,202604,202604,...,Dados,None,2026-05-13 16:10:20,680350,selecionado,"Selecionado por competencia do dado, pasta, no...",pronaf_gaia_202605131604.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,693dfe8836497391137942553d505a89241b7ab84cf091...,2026-05-14 16:35:44
3,mais_alimentos,Mais Alimentos,True,mais_alimentos_gaia_202605131504.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202605,202604,202604,...,Dados,None,2026-05-13 15:04:44,203644,selecionado,"Selecionado por competencia do dado, pasta, no...",mais_alimentos_gaia_202605131504.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,67ed3d3f66136da66c06f7626538f8e0a827448330b430...,2026-05-14 16:35:44
4,garantia_safra,Garantia-Safra,True,GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,...,DADOS,None,2026-05-05 12:48:06,96842,selecionado,"Selecionado por competencia do dado, pasta, no...",GARANTIA-SAFRA_2023-2024_ate_2026_04.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,608a621f836714aab73b2314f4557b0bdf6321b540b3b6...,2026-05-14 16:35:44
5,pncf,PNCF,True,pncf_2026_04_gerado_em_20260514095754.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202604,202604,202604,202604,...,DADOS,None,2026-05-14 09:57:54,13922,selecionado,"Selecionado por competencia do dado, pasta, no...",pncf_2026_04_gerado_em_20260514095754.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,90dfca8558e5030a45dbda7ece90fd5d77fe232c86d563...,2026-05-14 16:35:44
6,pnra,PNRA,True,PNRA_2026_2026_04_15.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,C:\Users\marce\OneDrive - Ministério da Agricu...,202603,202604,202604,202604,...,DADOS,None,2026-05-05 12:48:06,306820,selecionado,"Selecionado por competencia do dado, pasta, no...",PNRA_2026_2026_04_15.xlsx,C:\Users\marce\OneDrive - Ministério da Agricu...,c733b9b6b2def7f5096b391299c1d8504ee6b524ea824c...,2026-05-14 16:35:44


## Proximo passo

Se a preparacao terminou sem erro, execute o notebook `010_extracao_transformacao.ipynb`.

O `010` prioriza automaticamente `dados_brutos/dado_atual/dados_atuais` quando houver planilhas `.xlsx` nessa pasta.